# पाठ ०७ - योजना डिजाइन ढाँचा

यो नोटबुकले माइक्रोसफ्ट एजेन्ट फ्रेमवर्क प्रयोग गरी AI एजेन्टहरूको लागि **योजना डिजाइन ढाँचा** देखाउँछ।
तपाईंले जटिल यात्रा अनुरोधहरूलाई संरचित सानातिना कामहरूमा कसरी विभाजन गर्ने, तिनीहरूलाई विशेषज्ञ एजेन्टहरूलाई कसरी सुम्पने,
र संरचित आउटपुट प्रयोग गरेर परिणामस्वरूप योजना कसरी कार्यान्वयन गर्ने सिक्नुहुनेछ — सबै पाइडान्टिक मोडेलहरूले समर्थित।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## कार्य विघटन

कार्य विघटन योजना डिजाइन ढाँचाको मूल हो। एउटा एजेन्टलाई जटिल अनुरोध अन्त-देखि-अन्त समाधान गर्न भन्नुको सट्टा,
हामी समस्यालाई साना, स्पष्ट परिभाषित **उपकार्यहरू** मा विभाजन गर्छौं।
प्रत्येक उपकार्यलाई विशेषज्ञ एजेन्ट (जस्तै, उडानहरू, होटलहरू, गतिविधिहरू) लाई स्पष्ट
प्राथमिकता र निर्भरता क्रमसहित सुम्पिन्छौं।

यस दृष्टिकोणले केहि फाइदाहरू प्रदान गर्दछ:
- **स्पष्टता**: प्रत्येक उपकार्यको एक मात्र जिम्मेवारी हुन्छ।
- **समांतरता**: स्वतन्त्र उपकार्यहरू एकसाथ सञ्चालन गर्न सक्छन्।
- **विश्वसनीयता**: असफलताहरू व्यक्तिगत उपकार्यहरूमा सीमित हुन्छन्।
- **बजेट ट्र्याकिङ**: खर्चहरू उपकार्य अनुसार अनुमानित र जम्मा गरिन्छ।


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## संरचित आउटपुट सहित योजना बनाउने एजेन्ट सिर्जना गर्दै

योजना बनाउने एजेन्टले **फ्रन्ट डेस्क समन्वयकर्ता** को रूपमा काम गर्छ। उच्च-स्तरीय यात्रा अनुरोध दिँदा यो
संरचित `TravelPlan` उत्पादन गर्छ — अनुरोधलाई उपकार्यहरूमा विभाजन गर्दै, प्राथमिकताहरू निर्धारण गर्दै,
र आश्रितताहरू पत्ता लगाउँदै जसले कंसीयर्ज वा कार्यान्वयन तहले काम गर्न सक्छ।


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## विशेषज्ञ उपकरणहरूसँग योजना कार्यान्वयन गर्ने

फ्रन्ट डेस्क एजेन्टले संरचित योजना तयार गरेपछि, **कन्सियर्ज एजेन्ट** त्यसलाई कार्यान्वयन गर्छ।
प्रत्येक विशेषज्ञ उपकरणले एउटा उपकार्य वर्ग (उडान, होटल, गतिविधि) सम्हाल्दछ। कन्सियर्ज
योजनाका उपकार्यहरू निर्भरता क्रममा पुनरावृत्ति गर्दछ र प्रत्येकलाई
उपयुक्त उपकरणमा पठाउँछ।


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Summary

In this lesson you learned the **Planning Design Pattern** for AI agents:

1. **Task Decomposition** — A front desk planning agent breaks a complex travel request into
   structured subtasks using Pydantic models, assigning each to a specialist agent with priorities
   and dependencies.
2. **Structured Output** — By passing a `response_format` the agent returns a validated
   `TravelPlan` object instead of free-form text, making downstream processing reliable.
3. **Plan Execution** — A concierge agent iterates through the subtasks using specialist tools
   (`book_flight`, `reserve_hotel`, `book_activity`) to carry out the plan and report results.

This pattern separates *what to do* (planning) from *how to do it* (execution), making agents
more modular, testable, and easier to extend.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
